# Integrating a Bayesian optimizer with Giulia

This notebook describes the general pattern for exposing Giulia as the evaluator of a
candidate configuration. It is the reference for Section 4.3 of the report: the classes here
contain no scenario-specific quantity, and a study is expected to reuse them by supplying the
four definitions of the next section.

A concrete realization is given at the end as an example, not as the interface itself. The
companion notebook `rl_integration.ipynb` applies the same four definitions to a sequential
decision process.

## The four definitions an integration supplies

Chapter 2 describes the two interfaces through which an external component interacts with
Giulia: the **read path**, in which each module declares the quantities it exposes through
`SaveAble`, and the **write path**, in which Dedalo applies configuration changes to the
engineering-parameter tables.

Any over-the-top integration is fully specified by four definitions expressed against those
two interfaces:

1. **Observed quantities** — which exposed keys are read, and how they are aggregated.
2. **Modifiable parameters** — which engineering-parameter columns are written, and the
   admissible values of each.
3. **Point of application** — the stage of the simulation lifecycle at which a change takes
   effect.
4. **Objective** — how the measured quantities are converted into a reward or a score.

The difference with respect to the reinforcement-learning integration is not in these four
definitions but in what the external procedure receives. A policy is offered one transition
at a time; an optimizer is offered one configuration and the single number that summarizes
its evaluation. The simulator side is otherwise identical.


## The write path

In [ ]:
"""A minimal Dedalo subclass.

Identical to the one used for the reinforcement-learning integration: the write path does
not depend on the decision mechanism placed above it.
"""

from dedalo.dedalo import Dedalo


class OttDedalo(Dedalo):

    def process_static(self, episode_index: int = 0, **kwargs) -> None:
        """Applied once per episode, before `configure()`."""
        return None

    def process_realTime(self, step_index: int = 0, **kwargs) -> None:
        """Applied once per step, before `run_simulation()`."""
        return None

## Point of application

One objective evaluation is not one simulation step. The sequence below is written once in
the base class, so that the two quantities cannot be confused when a budget is reported:

| stage | call | repeated |
|---|---|---|
| `process_static(episode_index)` | cell-side parameters | once per episode |
| `apply_configuration(x)` | the candidate configuration | once per episode |
| `configure(episode_index, ...)` | builds the UE and link side | once per episode |
| `process_realTime(step_index)` | UE-side parameters | once per step |
| `run_simulation(step_index, ...)` | advances the engine | once per step |

The episode index is passed to `process_static`. Requesting several episodes therefore
produces several configured realizations rather than repetitions of the first one, which is
the condition for interpreting the aggregate as an average over traffic conditions.


## The evaluator

The evaluation sequence and the budget counters are written once. The four definitions are
the only methods a study overrides.

In [ ]:
"""A generic evaluator exposing Giulia as a black-box objective.

The evaluation sequence is fixed. A study supplies the four definitions by overriding
`apply_configuration` and `objective`, and by listing the keys it reads.
"""

from typing import Any, Dict, List

import numpy as np
from giulia.outputs.saveable import Saveable


class GiuliaEvaluator:

    def __init__(self, giulia, dedalo, *,
                 episodes: int = 1,
                 steps_per_episode: int = 10,
                 decision_interval_us: float = 1e6,
                 max_simulation_time_us: float | None = None,
                 observed_keys: List[str] | None = None):
        self.giulia = giulia
        self.dedalo = dedalo
        self.episodes = episodes
        self.steps_per_episode = steps_per_episode
        self.decision_interval_us = decision_interval_us
        # The engine stops executing events once the clock reaches the horizon given
        # to `configure()`. A horizon shorter than the requested intervals silently
        # produces empty steps, so the default is derived from the step budget.
        self.max_simulation_time_us = (
            max_simulation_time_us if max_simulation_time_us is not None
            else decision_interval_us * steps_per_episode)
        self.observed_keys = observed_keys or []
        # budget accounting, kept separate from the optimizer's own counters
        self.n_evaluations = 0
        self.n_episodes_run = 0
        self.n_steps_run = 0
        self._samples: List[Dict[str, Any]] = []

    # ---- the four definitions ------------------------------------------
    def apply_configuration(self, x: np.ndarray) -> None:
        """Write the candidate configuration through Dedalo."""
        raise NotImplementedError

    def read_sample(self) -> Dict[str, Any]:
        """Read the exposed quantities for the interval just completed."""
        return {key: Saveable.get(key) for key in self.observed_keys}

    def objective(self, samples: List[Dict[str, Any]]) -> float:
        """Aggregate the collected samples into one score. Larger is preferable."""
        raise NotImplementedError

    # ---- the fixed evaluation sequence ---------------------------------
    def evaluate(self, x: np.ndarray) -> float:
        self._samples = []
        for episode_index in range(self.episodes):
            self.dedalo.process_static(episode_index=episode_index)
            self.apply_configuration(x)
            self.giulia.configure(episode_index, self.max_simulation_time_us)
            for step_index in range(self.steps_per_episode):
                self.dedalo.process_realTime(step_index=step_index)
                self.giulia.run_simulation(step_index, self.decision_interval_us)
                self._samples.append(self.read_sample())
                self.n_steps_run += 1
            self.n_episodes_run += 1
        self.n_evaluations += 1
        return float(self.objective(self._samples))

## An instance

The following subclass searches over the activity of the terrestrial capacity cells and
scores a configuration by a weighted difference between a service term and an energy term.
The weight is a property of the study, not of the simulator: changing it changes which
configurations are preferred while the measured network response is unchanged.

In [ ]:
"""An instance: binary terrestrial carrier activity.

Only the four definitions are supplied. The evaluation sequence and the budget accounting
are inherited unchanged.
"""

import numpy as np
from giulia.outputs.saveable import Saveable


class CarrierActivityEvaluator(GiuliaEvaluator):

    ACTION_COLUMN = "radio_unit_initial_status"

    def __init__(self, giulia, dedalo, controlled_cells: list[str],
                 alpha: float = 0.5, **kwargs):
        super().__init__(
            giulia, dedalo,
            observed_keys=["BaseStations.prb_usage_per_cell",
                           "BaseStations.cell_radio_unit_statuses"],
            **kwargs)
        self.controlled_cells = controlled_cells
        self.alpha = alpha

    @property
    def dimension(self) -> int:
        return len(self.controlled_cells)

    # 2. modifiable parameters
    def apply_configuration(self, x: np.ndarray) -> None:
        states = ["active" if bit else "carrier_shutdown"
                  for bit in np.asarray(x).ravel().astype(int)]
        self.dedalo.set_giulia_network_df_ep(
            name_cell_list=self.controlled_cells,
            name_attribute=self.ACTION_COLUMN,
            value_list=states)

    # 4. objective: larger is preferable, both terms in [0, 1]
    def objective(self, samples) -> float:
        usage = np.array([np.asarray(s["BaseStations.prb_usage_per_cell"], dtype=float).mean()
                          for s in samples])
        active = np.array([(np.asarray(s["BaseStations.cell_radio_unit_statuses"],
                                       dtype=str) == "active").mean()
                           for s in samples])
        service = float(np.clip(usage.mean(), 0.0, 1.0))
        energy = float(np.clip(active.mean(), 0.0, 1.0))
        return (1.0 - self.alpha) * service - self.alpha * energy

## The optimizer is external

In [ ]:
"""A surrogate-agnostic optimization loop.

The loop separates three responsibilities: the initial design, the proposal rule, and the
evaluation. Only the third reaches the simulator, and it does so exclusively through
`evaluator.evaluate`. Replacing the proposal rule therefore changes the optimizer without
touching the integration.
"""

from typing import Callable, List, Tuple

import numpy as np


def run_optimization(evaluator,
                     propose: Callable[[np.ndarray, np.ndarray], np.ndarray],
                     n_initial: int,
                     n_guided: int,
                     seed: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    d = evaluator.dimension

    seen: set[tuple[int, ...]] = set()
    X: List[np.ndarray] = []
    y: List[float] = []

    def record(candidate: np.ndarray) -> None:
        key = tuple(int(v) for v in candidate)
        if key in seen:
            return
        seen.add(key)
        X.append(candidate.astype(float))
        y.append(evaluator.evaluate(candidate))

    # initial design
    while len(X) < n_initial:
        record(rng.integers(0, 2, size=d))

    # model-guided iterations
    for _ in range(n_guided):
        candidate = propose(np.asarray(X), np.asarray(y))
        record(candidate)

    return np.asarray(X), np.asarray(y)


def random_proposal(X: np.ndarray, y: np.ndarray, *, d: int, rng) -> np.ndarray:
    """The reference against which a surrogate-based rule is compared."""
    return rng.integers(0, 2, size=d)

## One proposal rule

In [ ]:
"""One proposal rule: a Gaussian-process surrogate with expected improvement.

BoTorch is installed in the project environment, so this cell runs. It is nevertheless one
choice among several: the loop above only requires a callable mapping the observed history
to the next candidate.
"""

import numpy as np
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from gpytorch.mlls import ExactMarginalLogLikelihood


def gp_proposal(X: np.ndarray, y: np.ndarray, *, d: int, rng, pool_size: int = 2048):
    """Fit a GP to the history and return the pooled candidate with the best acquisition."""
    train_x = torch.tensor(X, dtype=torch.double)
    train_y = torch.tensor(np.asarray(y), dtype=torch.double).unsqueeze(-1)

    model = SingleTaskGP(train_x, train_y)
    fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood, model))

    acquisition = qLogExpectedImprovement(model, best_f=train_y.max())

    pool = rng.integers(0, 2, size=(pool_size, d)).astype(float)
    with torch.no_grad():
        scores = acquisition(torch.tensor(pool, dtype=torch.double).unsqueeze(1))
    return pool[int(torch.argmax(scores))]


# A self-contained check of the proposal rule, on a synthetic objective rather than on
# Giulia, so that the mechanism can be exercised without a simulation build.
rng = np.random.default_rng(0)
d = 8
X0 = rng.integers(0, 2, size=(12, d)).astype(float)
y0 = X0.sum(axis=1) + 0.01 * rng.normal(size=12)
candidate = gp_proposal(X0, y0, d=d, rng=rng)
print("proposed candidate:", candidate.astype(int))

## Reporting the cost

In [ ]:
"""Budget accounting.

The evaluator counts simulator work separately from optimizer iterations, because the two
are not interchangeable: one objective evaluation contains `episodes x steps_per_episode`
simulation intervals plus one scenario configuration per episode.
"""

def report_budget(evaluator) -> None:
    print(f"objective evaluations : {evaluator.n_evaluations}")
    print(f"episodes configured   : {evaluator.n_episodes_run}")
    print(f"simulation intervals  : {evaluator.n_steps_run}")
    if evaluator.n_evaluations:
        print(f"intervals per evaluation: "
              f"{evaluator.n_steps_run / evaluator.n_evaluations:.1f}")

## Executing the search

In [ ]:
"""Running it.

The cell below builds a real Giulia instance and performs a small search. It is the only
expensive cell in this notebook: every recorded candidate is a full evaluation, so the
simulator cost is `n_initial + n_guided` times the cost of one evaluation.
"""

RUN = False   # set to True to execute

if RUN:
    import numpy as np
    from functools import partial
    from giulia.presets.g_standard_preset import Giulia

    g = Giulia("GiuliaStd", "3GPPTR38_901_4G_5G_multilayer", None,
               "uniform", None, "downlink", False, False, False, 1, False, False)
    dedalo = OttDedalo(g)

    cells = g.network_deployment_obj.df_ep.loc[
        g.network_deployment_obj.df_ep["cell_role"] == "capacity", "name"].tolist()

    evaluator = CarrierActivityEvaluator(
        g, dedalo, controlled_cells=cells, alpha=0.5,
        episodes=1, steps_per_episode=2)

    rng = np.random.default_rng(0)
    propose = partial(gp_proposal, d=evaluator.dimension, rng=rng)

    X, y = run_optimization(evaluator, propose, n_initial=4, n_guided=2, seed=0)

    print("best score:", y.max())
    report_budget(evaluator)

## What a study changes

| definition | where it is supplied | in the example above |
|---|---|---|
| observed quantities | `observed_keys`, `read_sample` | per-cell PRB usage and radio status |
| modifiable parameters | `apply_configuration` | `radio_unit_initial_status` on the capacity cells |
| point of application | inherited | `process_realTime` before each `run_simulation` |
| objective | `objective` | weighted service minus normalized activity |

The proposal rule, the surrogate and the acquisition criterion sit entirely outside this
table. Replacing `gp_proposal` with `random_proposal` changes the optimizer and leaves the
integration untouched, which is what makes the two comparable on the same evaluation
budget.